# CS436 / EE5310 / EE513 — Fall 2025
## PA3 (Part 2) Task 1 — Multi-Camera Calibration & Ground-Truth Reprojection - (30)

---

### **Objective**
In this part of PA3, you will work with the **EPFL Terrace multi-camera dataset** to explore real-world camera calibration and projection geometry.  
You will:
- Parse **Tsai calibration files** to extract intrinsic and extrinsic parameters  
- Compute **vanishing points** and **horizon lines**  
- Convert **ground-truth grid coordinates** to world coordinates  
- **Reproject** world points into image space using \( P = K[R|t] \)  
- Perform **decomposition**, **error analysis**, and **3D visualization** of camera models  

---

### **Instructions**
- Submission name: `RollNumber_PA3_Part2.ipynb`  
  *(Example: `26100277_PA3_Part2.ipynb`)*  
- **All cells must be executed** before submission; missing outputs will reduce marks.  
- Only the code written **within this notebook** will be graded — no external scripts.  
- Follow **good coding practices**:
  - Use clear variable names  
  - Add logical comments  
  - Keep plots neat and labeled  
- Maintain **academic integrity** — plagiarism or code sharing will result in disciplinary action.

---


## Step 1: Dataset & Resources

You will use the **EPFL Terrace Multi-Camera Dataset** available from the EPFL CVLab website:  
🔗 [https://www.epfl.ch/labs/cvlab/data/data-pom-index-php/](https://www.epfl.ch/labs/cvlab/data/data-pom-index-php/)

### Files Required
- **Calibration Files:** `terrace-tsai.zip`  
- **Ground Truth File:** [`gt_terrace1.txt`](https://www.epfl.ch/labs/cvlab/wp-content/uploads/2018/08/gt_terrace1.txt)  
- **Raw Videos:** `terrace_cam{0..3}.avi`  

---

## Setup Instructions

Before starting:
1. Mount your **Google Drive** if you are using Google Colab.  
2. Create the following folder structure inside your Drive:



In [ ]:
import os
import urllib.request

# Ensure base folder exists
RAW_DIR = f"{ROOT}/data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

# URLs for the four camera feeds (Terrace Sequence 1)
files = {
    "terrace_cam0.avi": "https://documents.epfl.ch/groups/c/cv/cvlab-pom-video3/www/terrace1-c0.avi",
    "terrace_cam1.avi": "https://documents.epfl.ch/groups/c/cv/cvlab-pom-video3/www/terrace1-c1.avi",
    "terrace_cam2.avi": "https://documents.epfl.ch/groups/c/cv/cvlab-pom-video3/www/terrace1-c2.avi",
    "terrace_cam3.avi": "https://documents.epfl.ch/groups/c/cv/cvlab-pom-video3/www/terrace1-c3.avi"
}

# Download any missing files
for name, url in files.items():
    dest = f"{RAW_DIR}/{name}"
    if not os.path.exists(dest):
        print(f"Downloading {name} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"Saved to {dest}")
    else:
        print(f"Already exists: {dest}")

print("\nAll Terrace Seq1 videos stored in Google Drive.")


## Step 2: Download Tsai Calibration Files

The Tsai calibration XMLs contain intrinsic (`K`), rotation (`R`), and translation (`t`) parameters for each of the 4 Terrace cameras.  
These parameters define the geometric relationship between **3D world coordinates** and **2D image projections**.

We will now download and extract the official EPFL-provided calibration files:  
🔗 [terrace-tsai.zip](https://www.epfl.ch/labs/cvlab/wp-content/uploads/2018/08/terrace-tsai.zip)


In [ ]:
# ===============================
# STEP 1: Download Tsai Calibration ZIP
# ===============================

import os, urllib.request, zipfile

CALIB_DIR = f"{ROOT}/data/calib"
os.makedirs(CALIB_DIR, exist_ok=True)

tsai_zip_url = "https://www.epfl.ch/labs/cvlab/wp-content/uploads/2018/08/terrace-tsai.zip"
tsai_zip_path = f"{CALIB_DIR}/terrace-tsai.zip"

if not os.path.exists(tsai_zip_path):
    print("⬇️ Downloading Tsai calibration ZIP ...")
    urllib.request.urlretrieve(tsai_zip_url, tsai_zip_path)
    print(f"✅ Saved to: {tsai_zip_path}")
else:
    print(f"⚡ Already exists: {tsai_zip_path}")

# Extract XML files
with zipfile.ZipFile(tsai_zip_path, 'r') as z:
    print("📦 Extracting contents:")
    z.extractall(CALIB_DIR)
    for name in z.namelist():
        print("  •", name)

print(f"\n✅ Tsai calibration files extracted to: {CALIB_DIR}")


## Step 3: Display Sample Frames
Load and display a few sample frames from the video feeds to visually inspect alignment and calibration quality.


In [ ]:
# Implement below

## Step 4: Compute Vanishing Points & Horizon Lines

Use the camera matrix **\( P \)** to find the **vanishing points** for the X, Y, and Z axes.  

The **horizon line** can be computed as:  
\[
L_{\infty} = P^{-T} [0, 0, 1, 0]^T
\]

Plot the vanishing points and horizon lines for each camera to verify calibration consistency.


In [ ]:
# Implement below

## Step 5: Load and Parse Ground-Truth File
Parse `gt_terrace1.txt` to extract:
- Number of frames
- Number of people
- Grid dimensions
- Frame step size  
Convert all valid grid positions to a DataFrame.


In [ ]:
# Download GT file
url = "https://www.epfl.ch/labs/cvlab/wp-content/uploads/2018/08/gt_terrace1.txt"
GT_PATH = f"{ROOT}/data/groundtruth/gt_terrace1.txt"
urllib.request.urlretrieve(url, GT_PATH)
print("✅ Downloaded:", GT_PATH)

# Parse file
with open(GT_PATH, "r") as f:
    _ = f.readline()
    header = f.readline().strip().split()
    body = [line.strip().split() for line in f if line.strip()]

num_frames, num_people, grid_w, grid_h, step, f0, f1 = map(int, header)
print(f"Frames={num_frames}, People={num_people}, Grid={grid_w}x{grid_h}, Step={step}")

rows = []
frame_idx = f0
for line in body:
    if len(line) < num_people: continue
    for pid, pos in enumerate(line[:num_people]):
        rows.append({"frame": frame_idx, "id": pid, "pos": int(pos)})
    frame_idx += step

gt_raw = pd.DataFrame(rows)
print("✅ Parsed GT:", gt_raw.shape)


## Step 6: Grid → World Coordinate Conversion
Convert valid grid cell indices to **top-view world coordinates (mm)** using the terrace parameters (You can get these from the EPFL website given above):
- Grid: 30 × 44  
- Origin: (-500, -1500)  
- Dimensions: (7500 × 11000)


In [ ]:
# Implement below

## Step 7: Ground-Truth Trajectory Visualization
Select one person and plot their trajectory(given in ground truth) on the top-view.


In [ ]:
# Implement below

## Step 8: Reprojection onto Image View
Reproject ground-truth world coordinates onto one camera image.


In [ ]:
# Implement below

## Step 9: Decomposition of Camera Matrix 
Decompose the camera matrix into K , R and t:
\[
P = K [R | t]
\]


In [ ]:
# Implement below

## Step 10: Reprojection and Individual Error Analysis
Compute and visualize the pixel-level reprojection error. Plot the original trajectory and the reprojected points and calculate reprojection error.
Compare the individual features K, R, T etc. Do error analysis and briefly explain the results.


In [ ]:
# Implement below

## Camera Model Visualization (Before vs After)
Compare Tsai and decomposed cameras in 3D figure and briefly explain results.


In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_title("Camera Models — Tsai vs Decomposed")
ax.set_xlabel("X (mm)"); ax.set_ylabel("Y (mm)"); ax.set_zlabel("Z (mm)")
scale = 500

for i, (R_tsai, t_tsai, P) in enumerate(zip(Rs, ts, Ps)):
    C_tsai = (-R_tsai.T @ t_tsai).flatten()
    K_d, R_d, t_h, *_ = cv2.decomposeProjectionMatrix(P)
    K_d /= K_d[2, 2]; t_d = (t_h / t_h[3])[:3]
    C_dec = (-R_d.T @ t_d).flatten()

    ax.scatter(*C_tsai, color='k', s=40, label='Tsai' if i==0 else "")
    ax.scatter(*C_dec, color='orange', s=40, label='Decomposed' if i==0 else "")
    ax.text(*C_tsai, f"Cam{i}")

ax.legend()
ax.view_init(25, 65)
plt.show()


## Task 02: Height Estimation using Known Height (20 Marks)

**Objective:**  
Estimate the height of an unknown object in an image using a reference object of known height, following *Algorithm 8.1*.

---

### Instructions:
- Select an image containing **two upright objects**, one with a **known real-world height** (e.g., a chair, desk, or door).  
- Using the provided known height as your reference, **apply Algorithm 8.1** to compute the height of the second object.  
- Show all steps clearly — label points on the image, show all intermediate calculations, and state any assumptions.  
- You may assume the objects are **on the same ground plane** and **ignore perspective distortion**.

---

### Expected Deliverables:
1. Input image (with key points marked).  
2. Computed scale factor based on the reference object.  
3. Estimated height of the second object (in same units).  
4. Brief explanation of steps and assumptions.


In [ ]:
# STEP 1 — Define known height (in millimeters)
known_height_mm = 1700 # e.g., height of a reference object in mm


# STEP 2 — Define pixel coordinates (x, y)
# Coordinates of top and bottom points of both objects in the image

# Reference object (known height)
ref_top = np.array([____, ____])
ref_base = np.array([____, ____])

# Unknown object (height to estimate)
obj_top = np.array([____, ____])
obj_base = np.array([____, ____])


# STEP 3 — Load and display the image
image_path = "path_to_your_image.jpg"   # replace with your image path
img = cv2.imread(image_path)
if img is None:
    raise FileNotFoundError("Image not found. Check the file path.")

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# STEP 4 — Compute pixel heights of both objects


# STEP 5 — Estimate the unknown object’s height
# (Use Algorithm 8.1)
# Write your formula below to compute the estimated height in meters



# STEP 6 — Display results and visualization
print(f"Reference object pixel height: {ref_height_px:.2f}px")
print(f"Unknown object pixel height: {obj_height_px:.2f}px")
print("Estimated Height of Unknown Object (in meters):", estimated_height_m)

plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)

for pt, color in [
    (ref_top, 'g'), (ref_base, 'g'),
    (obj_top, 'r'), (obj_base, 'r')
]:
    plt.plot(pt[0], pt[1], 'o', color=color, markersize=6)

plt.title("Points Marked for Height Estimation")
plt.axis('off')
plt.show()
